# Paper Disruption (CD) + F/E/G + ni/nj/nk — Numba engine

For every paper we compute, per citation window $W\in\{3,5,10,\text{all}\}$:
- **Disruption index** $CD=\dfrac{n_i-n_j}{n_i+n_j+n_k}$, where $n_i$ = citers of the focal paper $P$
  that cite **none** of $P$'s references, $n_j$ = citers of $P$ that **also** cite $\ge 1$ reference of
  $P$, $n_k$ = papers citing $\ge 1$ reference of $P$ but **not** $P$ itself.
- **F/E/G** (Foundation / Extension / Generalization): for each citer $c$, `up`=$|refs(P)\cap refs(c)|$
  and `down`=$|citers_W(P)\cap refs(c)|$; `up>down`→Extension, `down>up`→Foundation, `up=down=0`→
  Generalization, tie→½/½. Reported as fractions summing to 1.
- **ni / nj / nk** counts per window.

## Raw / input data
```
/project/jevans/Dawoon/Science of Science/OpenAlex/cache/paper_graph.npz   # citation edges (built by paper_citation): c_from, c_to (int32 CODE space), year, uni_mag
/project/jevans/Dawoon/Science of Science/OpenAlex/cache/paper_csr.npz     # derived int32 CSR cache (out/in adjacency) -- built here on first run
```
Codes are positions in `uni_mag` (sorted MAG ids); `paper_id = W{mag}`.

## Engine
A `numba` `njit(parallel=True)` kernel walks each focal paper's citers and references over an **int32
CSR** (references = out-edges, citers = in-edges), doing the two-hop set operations with local
binary-search membership (no huge per-paper Python objects). The CSR is cached so reruns skip the
(one-time) build. Validated exact against a numpy per-focal oracle and the independent year-2000 check.

## Output
`/project/jevans/Dawoon/Science of Science/OpenAlex/output/paper_disruption.parquet` — `paper_id` + `CD/F/E/G/ni/nj/nk` × `{_3,_5,_10,_all}`
(28 metric columns). Papers with no citers are left NaN (CD/F/E/G) / -1 (ni/nj/nk).

In [ ]:
import os, sys, gc, time
import numpy as np, pandas as pd
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/OpenAlex')
import oa_common as oa
ROOT = oa.BASE; OUT = oa.OUT
print('snapshot:', oa.ROOT)
OUT_FP = f'{OUT}/paper_disruption.parquet'
WINS = np.array([3, 5, 10, 2_000_000_000], dtype=np.int64)
SFX  = ['_3', '_5', '_10', '_all']
# Everything below is fed by notebook/referenced_works_w_year.ipynb: it writes the per-work
# map (year + source) and the edge table with both years and both source ids, and
# oa.load_graph() / oa.load_csr() / oa.build_journal() read those instead of re-walking
# renli's tree. Build it once before running this notebook.
assert oa.have_consolidated(), (
    'run notebook/referenced_works_w_year.ipynb first — it builds the map and edge table')
oa.summary()

## 1. Load int32 CSR (build from the citation graph + cache on first run)

In [ ]:
%%time
out_ptr, out_idx, in_ptr, in_idx, year, uni_mag = oa.load_csr()   # builds on first run
n = len(year)
print(f'CSR: {n:,} papers, {len(out_idx):,} edges')

## 2. Numba engine (validated: 0 mismatches vs oracle; ni/nj/nk 100% vs indep. 2000 check)

In [ ]:
from numba import njit, prange

@njit(inline='always')
def _bfind(arr, x):
    lo = 0; hi = len(arr)
    while lo < hi:
        mid = (lo + hi) >> 1
        if arr[mid] < x: lo = mid + 1
        else: hi = mid
    return lo < len(arr) and arr[lo] == x

@njit(parallel=True)
def compute_numba(focal, out_ptr, out_idx, in_ptr, in_idx, year, wins,
                  CD, Ff, Ef, Gf, NI, NJ, NK):
    W = len(wins)
    for t in prange(len(focal)):
        F = focal[t]
        a0 = in_ptr[F]; a1 = in_ptr[F + 1]
        if a1 == a0:
            continue
        yF = year[F]
        A = in_idx[a0:a1]
        R = out_idx[out_ptr[F]:out_ptr[F + 1]]
        Rs = np.sort(R); As = np.sort(A)
        Nw = np.zeros(W, np.int64); njw = np.zeros(W, np.int64)
        cext = np.zeros(W, np.int64); cfnd = np.zeros(W, np.int64)
        tiew = np.zeros(W, np.int64); cgw = np.zeros(W, np.int64); Bw = np.zeros(W, np.int64)
        for ci in range(len(A)):
            c = A[ci]; dc = year[c] - yF
            if dc < 0:
                continue
            up = 0; downc = np.zeros(W, np.int64)
            for ri in range(out_ptr[c], out_ptr[c + 1]):
                d = out_idx[ri]
                if _bfind(Rs, d): up += 1
                if _bfind(As, d):
                    dd = year[d] - yF
                    if dd >= 0:
                        for k in range(W):
                            if dd <= wins[k]: downc[k] += 1
            for k in range(W):
                if dc <= wins[k]:
                    Nw[k] += 1; dk = downc[k]
                    if up > 0: njw[k] += 1
                    if up > dk: cext[k] += 1
                    elif dk > up: cfnd[k] += 1
                    elif up == dk and up > 0: tiew[k] += 1
                    elif up == 0 and dk == 0: cgw[k] += 1
        totB = 0
        for ri in range(len(R)):
            r = R[ri]; totB += in_ptr[r + 1] - in_ptr[r]
        if totB > 0:
            buf = np.empty(totB, np.int32); p = 0
            for ri in range(len(R)):
                r = R[ri]
                for j in range(in_ptr[r], in_ptr[r + 1]):
                    buf[p] = in_idx[j]; p += 1
            buf.sort(); prev = np.int32(-1)
            for ii in range(totB):
                b = buf[ii]
                if b == prev or b == F: continue
                prev = b; dd = year[b] - yF
                if dd >= 0:
                    for k in range(W):
                        if dd <= wins[k]: Bw[k] += 1
        for k in range(W):
            Nk = Nw[k]; Bk = Bw[k]
            if Nk == 0 and Bk == 0: continue
            njk = njw[k]; nik = Nk - njk; nkk = Bk - njk; denom = nik + njk + nkk
            NI[k, F] = nik; NJ[k, F] = njk; NK[k, F] = nkk
            CD[k, F] = (nik - njk) / denom if denom > 0 else np.nan
            if Nk > 0:
                Ef[k, F] = (cext[k] + 0.5 * tiew[k]) / Nk
                Ff[k, F] = (cfnd[k] + 0.5 * tiew[k]) / Nk
                Gf[k, F] = cgw[k] / Nk
print('engine ready')

## 3. Run compute over all focal papers (with citers), then save

In [ ]:
%%time
W = len(WINS)
CD = np.full((W, n), np.nan, np.float32); Ff = np.full((W, n), np.nan, np.float32)
Ef = np.full((W, n), np.nan, np.float32); Gf = np.full((W, n), np.nan, np.float32)
NI = np.full((W, n), -1, np.int32); NJ = np.full((W, n), -1, np.int32); NK = np.full((W, n), -1, np.int32)
focal_all = np.flatnonzero(in_ptr[1:] - in_ptr[:-1] > 0).astype(np.int64)
print(f'focal with citers: {len(focal_all):,}  -- warm-up compile...')
compute_numba(focal_all[:5000], out_ptr, out_idx, in_ptr, in_idx, year, WINS, CD, Ff, Ef, Gf, NI, NJ, NK)
CD[:] = np.nan; Ff[:] = np.nan; Ef[:] = np.nan; Gf[:] = np.nan; NI[:] = -1; NJ[:] = -1; NK[:] = -1
tc = time.time()
print('running full numba compute...')
compute_numba(focal_all, out_ptr, out_idx, in_ptr, in_idx, year, WINS, CD, Ff, Ef, Gf, NI, NJ, NK)
print(f'compute done in {time.time()-tc:.0f}s')

In [ ]:
cols = {'paper_id': np.char.add('W', uni_mag.astype(str))}
for k, s in enumerate(SFX):
    cols[f'CD{s}'] = CD[k]; cols[f'F{s}'] = Ff[k]; cols[f'E{s}'] = Ef[k]; cols[f'G{s}'] = Gf[k]
    cols[f'ni{s}'] = NI[k]; cols[f'nj{s}'] = NJ[k]; cols[f'nk{s}'] = NK[k]
out = pd.DataFrame(cols)
out.to_parquet(OUT_FP, index=False)
print(f'WROTE {OUT_FP}  ({len(out):,} rows, {len(out.columns)} cols)')
for k, s in enumerate(SFX):
    print(f'  CD{s}: defined {np.isfinite(CD[k]).mean()*100:5.1f}%  mean {np.nanmean(CD[k]):+.4f}  |  '
          f'f/e/g = {np.nanmean(Ff[k]):.3f}/{np.nanmean(Ef[k]):.3f}/{np.nanmean(Gf[k]):.3f}')
display(out.head(10))